# FibRec, FibMemo e FibIt (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/FibRec_FibMemo_FibIt_python.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `FibRec`, `FibMemo` e `FibIt`: gli algoritmi in Python, eseguiti sugli esempi del capitolo, con la cella «Prova tu» per provarli con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/FibRec_FibMemo_FibIt_cpp.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/FibRec_FibMemo_FibIt_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_python.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Python 3, senza installare nulla.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `fib_rec` (snake_case).

## Fibonacci: FibRec, FibMemo e FibIt (sottoproblemi che si sovrappongono)

*Input*: un naturale $n$. *Output*: il numero di Fibonacci $F(n)$, con $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — traduzione diretta della definizione: i sottoproblemi si sovrappongono e le chiamate sono $2F(n+1)-1=\Theta(\varphi^n)$, esponenziali; la profondità è $n$.
- **FibMemo**$(n,memo)$ — ricorsione con memoizzazione: ogni valore è calcolato una volta (il vettore `memo` vale $-1$ per «non definito»); $2n-1$ chiamate, memoria $\Theta(n)$.
- **FibIt**$(n)$ — ciclo con due variabili: $n-1$ iterazioni, memoria $\Theta(1)$.

Sull'esempio $\texttt{FibRec}(5)$ del capitolo si conta quante volte è chiamata ogni sottoproblema (l'albero ha $15$ nodi, ma servono solo $6$ valori diversi). Poi la tabella dei conteggi (che coincidono con i valori del capitolo: $21\,891$, $242\,785$, $2\,692\,537$ chiamate per $n=20,25,30$) e i tempi misurati.

In [1]:
import time

fib_calls = 0   # chiamate eseguite

def fib_rec(n):
    """FibRec(n)"""
    global fib_calls
    fib_calls += 1
    if n <= 1:
        return n
    return fib_rec(n - 1) + fib_rec(n - 2)

def fib_memo(n, memo):
    """FibMemo(n, memo): memo[k] = -1 significa «non definito»."""
    global fib_calls
    fib_calls += 1
    if n <= 1:
        return n
    if memo[n] == -1:
        memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

def fib_it(n):
    """FibIt(n)"""
    if n <= 1:
        return n
    prev = 0                        # F(i-2)
    curr = 1                        # F(i-1)
    i = 2
    while i <= n:
        nxt = prev + curr
        prev = curr
        curr = nxt
        i = i + 1
    return curr

def fib_rec_hist(n, counts):
    """Come FibRec, ma conta le chiamate per ogni valore del parametro."""
    counts[n] += 1
    if n <= 1:
        return n
    return fib_rec_hist(n - 1, counts) + fib_rec_hist(n - 2, counts)

counts = [0] * 6
print("FibRec(5) =", fib_rec_hist(5, counts))
print("chiamate per ogni valore del parametro:", "  ".join(f"{k}:{counts[k]}" for k in range(5, -1, -1)), f"  totale {sum(counts)}")
print()
print(f"{'n':>3s} {'F(n)':>8s} {'FibRec':>9s} {'2F(n+1)-1':>10s} {'FibMemo':>8s} {'FibIt':>6s}  (chiamate, chiamate, chiamate, iterazioni)")
for n in [10, 15, 20, 25, 30]:
    f = fib_it(n)
    fib_calls = 0
    r1 = fib_rec(n)
    c1 = fib_calls
    fib_calls = 0
    r2 = fib_memo(n, [-1] * (n + 1))
    c2 = fib_calls
    ok = "OK" if (r1 == r2 == f and c1 == 2 * fib_it(n + 1) - 1 and c2 == 2 * n - 1) else "ERROR"
    print(f"{n:>3d} {f:>8d} {c1:>9d} {2 * fib_it(n + 1) - 1:>10d} {c2:>8d} {n - 1:>6d}  {ok}")
print()
print("Tempi misurati (dipendono dalla macchina):")
for n in [20, 25, 30]:
    t0 = time.perf_counter(); fib_rec(n); t1 = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter()
    for _ in range(1000):
        fib_it(n)
    t2 = (time.perf_counter() - t0) * 1000 / 1000
    print(f"n={n:<3d} FibRec {t1:>10.4f} ms   FibIt {t2:>10.6f} ms")

FibRec(5) = 5
chiamate per ogni valore del parametro: 5:1  4:1  3:2  2:3  1:5  0:3   totale 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (chiamate, chiamate, chiamate, iterazioni)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Tempi misurati (dipendono dalla macchina):
n=20  FibRec     1.8652 ms   FibIt   0.000826 ms
n=25  FibRec    20.5861 ms   FibIt   0.001589 ms
n=30  FibRec   239.8623 ms   FibIt   0.001238 ms


**Prova tu.** Aumenta $n$ (con $n=32$ `FibRec` esegue $7\,049\,155$ chiamate, `FibMemo` $63$) e guarda come cresce il tempo di `FibRec`.

In [2]:
# Aumenta n: con n = 32 FibRec esegue 7 049 155 chiamate, FibMemo 63.
n = 32
fib_calls = 0
print(fib_rec(n), fib_calls)
fib_calls = 0
print(fib_memo(n, [-1] * (n + 1)), fib_calls)

2178309 7049155
2178309 63


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).